# VISIONAI GPU - the real one
Trains a much bigger DDPM (the same class of model behind pro image generators) from scratch on Google's free T4 GPU.

1. Runtime -> Change runtime type -> T4 GPU
2. Runtime -> Run all
3. Let it run ~3 hours (progress bar below)
4. Your weights + sample images auto-save to Google Drive

Result: actually recognizable airplanes, horses, ships... not pixel soup.


In [ ]:
import torch, os, math, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/VISIONAI'
except Exception:
    CKPT_DIR = '/content/VISIONAI'
os.makedirs(CKPT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected! Runtime -> Change runtime type -> T4 GPU, then Run all again.')


In [ ]:
T = 1000
EMB = 256
CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

def cosine_schedule(T=1000, s=0.008):
    t = torch.arange(T, dtype=torch.float64) / T
    f = torch.cos(((t + s) / (1 + s)) * math.pi / 2) ** 2
    return (f / f[0]).clamp(1e-8, 1)

class TimeEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        ang = t.float()[:, None] * freqs[None]
        return torch.cat([torch.sin(ang), torch.cos(ang)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, ic), nn.Conv2d(ic, oc, 3, padding=1)
        self.emb = nn.Linear(EMB, oc)
        self.n2, self.c2 = nn.GroupNorm(8, oc), nn.Conv2d(oc, oc, 3, padding=1)
        self.skip = nn.Conv2d(ic, oc, 1) if ic != oc else nn.Identity()
    def forward(self, x, emb):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.emb(F.silu(emb))[:, :, None, None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)

class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.norm = nn.GroupNorm(8, c)
        self.qkv = nn.Conv2d(c, c * 3, 1)
        self.out = nn.Conv2d(c, c, 1)
    def forward(self, x):
        b, c, h, w = x.shape
        q, k, v = self.qkv(self.norm(x)).reshape(b, 3, c, h * w).unbind(1)
        a = torch.softmax(q.transpose(1, 2) @ k / c ** 0.5, dim=-1)
        return x + self.out((a @ v.transpose(1, 2)).transpose(1, 2).reshape(b, c, h, w))

class UNet(nn.Module):
    def __init__(self, base=128):
        super().__init__()
        b = base
        self.temb = nn.Sequential(nn.Linear(EMB, EMB), nn.SiLU(), nn.Linear(EMB, EMB))
        self.cls = nn.Embedding(10, EMB)
        self.inc = nn.Conv2d(3, b, 3, padding=1)
        self.d1a, self.d1b = ResBlock(b, b), ResBlock(b, b)
        self.down1 = nn.Conv2d(b, b, 3, stride=2, padding=1)
        self.d2a, self.d2b = ResBlock(b, b * 2), ResBlock(b * 2, b * 2)
        self.down2 = nn.Conv2d(b * 2, b * 2, 3, stride=2, padding=1)
        self.m1, self.attn, self.m2 = ResBlock(b * 2, b * 2), Attn(b * 2), ResBlock(b * 2, b * 2)
        self.u2a, self.u2b = ResBlock(b * 4, b * 2), ResBlock(b * 2, b * 2)
        self.up2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.u1a, self.u1b = ResBlock(b * 3, b), ResBlock(b, b)
        self.up1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.onorm = nn.GroupNorm(8, b)
        self.outc = nn.Conv2d(b, 3, 3, padding=1)
    def forward(self, x, t, cls):
        emb = self.temb(TimeEmbed(EMB)(t)) + self.cls(cls)
        s1 = self.d1b(self.d1a(self.inc(x), emb), emb)
        h = self.down1(s1)
        s2 = self.d2b(self.d2a(h, emb), emb)
        h = self.down2(s2)
        h = self.m2(self.attn(self.m1(h, emb)), emb)
        h = self.up2(h)
        h = self.u2b(self.u2a(torch.cat([h, s2], 1), emb), emb)
        h = self.up1(h)
        h = self.u1b(self.u1a(torch.cat([h, s1], 1), emb), emb)
        return self.outc(F.silu(self.onorm(h)))

model = UNet(128).to(device)
ema = {k: v.detach().clone().float() for k, v in model.state_dict().items()}
print('parameters: %.1fM' % (sum(p.numel() for p in model.parameters()) / 1e6))


In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,) * 3, (0.5,) * 3)])
ds = datasets.CIFAR10('/content/data', train=True, download=True, transform=tf)
dl = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2, drop_last=True, persistent_workers=True)
print('60,000 real internet-collected photos loaded')
print('classes:', CLASSES)


In [ ]:
TARGET = 40000            # ~3h on a T4. Raise to 100000 for a sharper overnight run.
CKPT = os.path.join(CKPT_DIR, 'visionai_gpu.pt')
abar_all = cosine_schedule(T).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))
step = 0
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    ema.update(ck['ema']); step = ck['step']
    print('resumed from step', step)

pbar = None
try:
    from tqdm.auto import tqdm
    pbar = tqdm(total=TARGET, initial=step, desc='VISIONAI training')
except ImportError:
    pass
while step < TARGET:
    for x, y in dl:
        t = torch.randint(0, T, (x.shape[0],), device=device)
        noise = torch.randn_like(x)
        xt = abar_all[t].sqrt()[:, None, None, None] * x.to(device) \
             + (1 - abar_all[t]).sqrt()[:, None, None, None] * noise
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            pred = model(xt, t, y.to(device))
            loss = F.mse_loss(pred, noise)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            for k, v in model.state_dict().items():
                ema[k].mul_(0.999).add_(v.detach().float(), alpha=0.001)
        step += 1
        if pbar: pbar.update(1)
        if step % 500 == 0:
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                        'ema': ema, 'step': step}, CKPT)
        if step % 2000 == 0 and pbar:
            pbar.set_postfix(loss='%.3f' % loss.item())
        if step >= TARGET:
            break
if pbar: pbar.close()
print('TRAINING DONE ->', CKPT)


In [ ]:
@torch.no_grad()
def ddim_sample(n, cls_idx, steps=100):
    model.load_state_dict(ema)
    model.eval()
    x = torch.randn(n, 3, 32, 32, device=device)
    ts = torch.linspace(T - 1, 0, steps).long().to(device)
    for i in range(steps):
        t_now = ts[i]
        t_next = ts[i + 1] if i + 1 < steps else ts.new_zeros(())
        tb = t_now.repeat(n)
        eps = model(x, tb, torch.full((n,), cls_idx, device=device, dtype=torch.long))
        ab, ab_n = abar_all[t_now], abar_all[t_next]
        x0 = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-1, 1)
        x = ab_n.sqrt() * x0 + (1 - ab_n).sqrt() * eps
    return x

import matplotlib.pyplot as plt
rows = []
for ci in range(10):
    rows.append(ddim_sample(2, ci).cpu())
img = torch.cat(rows).clamp(-1, 1)
fig, ax = plt.subplots(10, 2, figsize=(3, 15))
for i in range(20):
    ax[i // 2, i % 2].imshow((img[i].permute(1, 2, 0) + 1) / 2)
    ax[i // 2, i % 2].axis('off')
plt.savefig(os.path.join(CKPT_DIR, 'visionai_grid.png'), dpi=200, bbox_inches='tight')
plt.show()
print('grid saved to', os.path.join(CKPT_DIR, 'visionai_grid.png'))


In [ ]:
from PIL import Image
import numpy as np

def make(cls_name, count=1, size=144):
    # generate any class: make('horse', 4, 144)
    ci = CLASSES.index(cls_name)
    out = ddim_sample(count, ci).cpu().clamp(-1, 1)
    paths = []
    for i in range(count):
        arr = ((out[i].permute(1, 2, 0).numpy() + 1) * 127.5).astype(np.uint8)
        p = os.path.join(CKPT_DIR, 'gen_%s_%d.png' % (cls_name, i))
        Image.fromarray(arr).resize((size, size), Image.BICUBIC).save(p)
        paths.append(p)
    plt.figure(figsize=(2 * count, 2))
    for i, p in enumerate(paths):
        plt.subplot(1, count, i + 1)
        plt.imshow(Image.open(p)); plt.axis('off')
    plt.show()
    return paths

# try it: make('horse', 4, 144)


## How to generate more
In the cell below, call `make('horse', 4, 144)` - class, count, size. Try every class!

Classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.
Want sharper images? Re-run the training cell with TARGET raised (e.g. 100000, overnight run).
It resumes from your Drive checkpoint automatically - it never loses progress.
